<div align="center">

# NB1 · The Black Box Problem in Medical AI

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · 28 October 2026 · Session 10:00 to 10:20

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB1_Black_Box_Problem.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

A diagnostic model that scores 97 percent accuracy sounds ready for the clinic. This notebook takes such a model apart and shows what the headline number hides: missed cancers, predictions that are wrong and confident at the same time, probabilities that do not mean what they appear to mean, and a model that quietly learns the hospital instead of the disease. The notebook closes with the three legal instruments that turn these observations into obligations.

The running example is the Wisconsin Diagnostic Breast Cancer dataset [1, 2], which ships with scikit-learn, so nothing has to be downloaded. The whole notebook runs in about a minute on a free Colab CPU. Cells with sliders are interactive; every other cell can be read as a lecture note.

**Companion tool.** During the session the same model is available in the browser, with no installation, in the *Deploy?* tab of the [XAI Lab](https://utkukose.github.io/xai-cars2026-NB-workshop/).

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss,
                             confusion_matrix, roc_auc_score)
from sklearn.calibration import calibration_curve
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. A model that scores 97 percent

Each of the 569 records describes cell nuclei in a digitised image of a fine needle aspirate of a breast mass. Thirty features summarise the radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry and fractal dimension of the nuclei [1]. scikit-learn codes malignant tumours as 0; the code below recodes them so that 1 means malignant, the class a clinician is worried about missing.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = (data.target == 0).astype(int).rename("malignant")
print(f"{len(X)} biopsies · {X.shape[1]} features · {y.sum()} malignant ({y.mean():.0%})")

The model is a small neural network, the kind of black box that often appears in published diagnostic studies. Performance is estimated with five-fold cross-validation, and every biopsy is scored by a model that never saw it during training. These out-of-fold predictions are the honest basis for everything that follows.

In [ ]:
mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=2000, random_state=SEED))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_mlp = cross_val_predict(mlp, X, y, cv=cv, method="predict_proba")[:, 1]
pred = (p_mlp >= 0.5).astype(int)

headline = pd.Series({
    "Accuracy": accuracy_score(y, pred),
    "Balanced accuracy": balanced_accuracy_score(y, pred),
    "AUROC": roc_auc_score(y, p_mlp),
}).round(3)
headline.to_frame("out-of-fold estimate")

> **Poll.** A hospital committee receives this table. Should the model be deployed to triage biopsies? *Yes / No / It depends on what else we know.*

## 2. What the headline number hides

Accuracy averages two very different kinds of mistake. A false negative sends a patient with cancer home; a false positive leads to an unnecessary work-up. The confusion matrix separates them, and the decision threshold decides how the errors are traded against each other. The threshold is a clinical and ethical choice, not a property of the model.

In [ ]:
def clinical_counts(threshold):
    pr = (p_mlp >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, sensitivity=tp / (tp + fn), specificity=tn / (tn + fp))


def show_threshold(threshold=0.5):
    c = clinical_counts(threshold)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
    bins = np.linspace(0, 1, 26)
    axes[0].hist(p_mlp[y == 0], bins=bins, color=SAFE, alpha=0.75, label="benign")
    axes[0].hist(p_mlp[y == 1], bins=bins, color=ALERT, alpha=0.75, label="malignant")
    axes[0].axvline(threshold, color=INK, lw=2, ls="--", label=f"threshold {threshold:.2f}")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("predicted probability of malignancy")
    axes[0].set_ylabel("biopsies (log scale)")
    axes[0].legend(frameon=False)
    labels = ["missed cancers\n(false negatives)", "unnecessary work-ups\n(false positives)"]
    axes[1].bar(labels, [c["fn"], c["fp"]], color=[ALERT, AMBER])
    for k, v in enumerate([c["fn"], c["fp"]]):
        axes[1].text(k, v, str(v), ha="center", va="bottom", fontsize=12, fontweight="bold")
    axes[1].set_title(f"sensitivity {c['sensitivity']:.3f} · specificity {c['specificity']:.3f}")
    axes[1].set_ylim(0, max(c["fn"], c["fp"], 5) * 1.3)
    plt.tight_layout()
    plt.show()


widgets.interact(show_threshold,
                 threshold=widgets.FloatSlider(value=0.5, min=0.02, max=0.98, step=0.02,
                                               continuous_update=False, description="threshold"));
c = clinical_counts(0.5)
print(f"At threshold 0.50: {c['fn']} of {c['fn'] + c['tp']} cancers missed, {c['fp']} unnecessary work-ups")

Even at the default threshold, the missed cancers in the right-hand panel are patients, not a rounding error in the accuracy figure. Lowering the threshold recovers them at the cost of more work-ups. No single number in the headline table reveals where a given hospital should sit on this curve.

## 3. Wrong and sure

The more troubling errors are the confident ones. A prediction of 0.97 for "benign" on a malignant tumour looks exactly like a correct prediction of 0.97. Nothing in the output warns the clinician.

In [ ]:
confidence = np.where(pred == 1, p_mlp, 1 - p_mlp)
wrong = pred != y.to_numpy()
errors = pd.DataFrame({
    "true class": np.where(y == 1, "malignant", "benign"),
    "P(malignant)": p_mlp.round(3),
    "confidence in the wrong answer": confidence.round(3),
})[wrong].sort_values("confidence in the wrong answer", ascending=False)
print(f"{wrong.sum()} errors in {len(y)} biopsies; "
      f"{(confidence[wrong] >= 0.9).sum()} of them with confidence of at least 0.90")
errors.head(8)

The plot below places the most confident error next to the two class averages. Each bar is the distance, in standard deviations, between the biopsy and the average benign or malignant biopsy for one feature. The model gives no reason for its answer; at this point only raw feature values can be inspected, and thirty of them do not add up to a clinical argument.

In [ ]:
Z = (X - X.mean()) / X.std()
worst = errors.index[0]
dev = pd.DataFrame({
    "vs benign average": Z.loc[worst] - Z[y == 0].mean(),
    "vs malignant average": Z.loc[worst] - Z[y == 1].mean(),
})
top = dev.abs().max(axis=1).sort_values(ascending=False).index[:10]
ax = dev.loc[top][::-1].plot.barh(figsize=(8, 4.2), color=[SAFE, ALERT], width=0.75)
ax.axvline(0, color=INK, lw=1)
ax.set_xlabel("standardised distance")
ax.set_title(f"Biopsy {worst}: truly {errors.loc[worst, 'true class']}, "
             f"predicted P(malignant) = {errors.loc[worst, 'P(malignant)']}")
plt.tight_layout()
plt.show()

## 4. Confidence is not correctness

A probability is useful only if it is calibrated: among all biopsies scored 0.8, about 80 percent should be malignant. Modern neural networks are often overconfident [3]. The reliability diagram compares the network with a plain logistic regression, and the expected calibration error (ECE) summarises the gap between stated and observed frequencies,

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{n_b}{N}\,\bigl|\bar{y}_b - \bar{p}_b\bigr|,$$

where $\bar{p}_b$ is the mean predicted probability and $\bar{y}_b$ the observed frequency of malignancy in bin $b$.

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=5000))
p_lr = cross_val_predict(logreg, X, y, cv=cv, method="predict_proba")[:, 1]


def expected_calibration_error(y_true, p, n_bins=10):
    y_true = np.asarray(y_true)
    idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)) - 1, 0, n_bins - 1)
    return sum((idx == b).mean() * abs(y_true[idx == b].mean() - p[idx == b].mean())
               for b in range(n_bins) if (idx == b).any())


fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot([0, 1], [0, 1], color=MUTED, ls=":", label="perfect calibration")
for name, p, col in [("neural network", p_mlp, ALERT), ("logistic regression", p_lr, SAFE)]:
    frac, mean_pred = calibration_curve(y, p, n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac, "o-", color=col,
            label=f"{name}: ECE {expected_calibration_error(y, p):.3f}, Brier {brier_score_loss(y, p):.3f}")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed frequency of malignancy")
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 5. The shortcut: when a model learns the hospital instead of the disease

Deep models for chest radiographs have been shown to recognise the hospital or the scanner that produced an image and to use it as a proxy for disease prevalence [4, 5]. A classic tabular case is the pneumonia model that learned asthma lowers mortality risk, because asthmatic patients were routinely sent to intensive care [6]. Such shortcuts pass internal validation and fail elsewhere.

The experiment below reproduces the mechanism in a controlled way. A subset of weaker morphometric features is used, and a binary variable `acquisition_site` is added. At the development hospital, malignant biopsies come mostly from a referral centre (site 1), so the site correlates with the diagnosis. At an external hospital the site says nothing about the diagnosis. Two gradient boosting models are compared: one that may use the site and one that may not.

In [ ]:
WEAK = ["mean texture", "mean smoothness", "mean symmetry", "mean fractal dimension",
        "texture error", "smoothness error", "symmetry error", "fractal dimension error"]
idx_dev, idx_ext = train_test_split(np.arange(len(X)), test_size=0.4, stratify=y, random_state=SEED)
site = np.zeros(len(X), dtype=int)
site[idx_dev] = (rng.random(len(idx_dev)) < np.where(y.to_numpy()[idx_dev] == 1, 0.9, 0.1)).astype(int)
site[idx_ext] = (rng.random(len(idx_ext)) < 0.5).astype(int)
X_site = X[WEAK].assign(acquisition_site=site)

idx_tr, idx_te = train_test_split(idx_dev, test_size=0.35, stratify=y.to_numpy()[idx_dev], random_state=SEED)
gb_site = GradientBoostingClassifier(random_state=SEED).fit(X_site.iloc[idx_tr], y.iloc[idx_tr])
gb_nosite = GradientBoostingClassifier(random_state=SEED).fit(X[WEAK].iloc[idx_tr], y.iloc[idx_tr])

def auroc(model, frame, idx):
    return roc_auc_score(y.iloc[idx], model.predict_proba(frame.iloc[idx])[:, 1])

pd.DataFrame({
    "internal test AUROC": [auroc(gb_site, X_site, idx_te), auroc(gb_nosite, X[WEAK], idx_te)],
    "external hospital AUROC": [auroc(gb_site, X_site, idx_ext), auroc(gb_nosite, X[WEAK], idx_ext)],
}, index=["model allowed to use acquisition_site", "model without acquisition_site"]).round(3)

The model that uses the site wins the internal comparison and loses at the external hospital, where the model without the site holds its level. An explanation of the first model, even the simplest one, exposes the problem before any external data exist. Permutation importance measures how much the internal test AUROC drops when one feature is shuffled.

In [ ]:
imp = permutation_importance(gb_site, X_site.iloc[idx_te], y.iloc[idx_te], scoring="roc_auc",
                             n_repeats=10 if QUICK else 30, random_state=SEED)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(7, 3.8))
colors = [ALERT if X_site.columns[i] == "acquisition_site" else INK for i in order]
ax.barh(X_site.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color=colors)
ax.set_xlabel("drop in internal test AUROC when the feature is shuffled")
ax.set_title("What the development-hospital model relies on")
plt.tight_layout()
plt.show()

## 6. Why a security audience should care

Three observations connect explainability to security. First, a shortcut is a backdoor that nobody planted: a feature that switches the decision without any clinical meaning, which an attacker who discovers it can exploit as easily as a planted trigger. Second, the non-robust features that shortcuts rely on are the same ones that adversarial examples exploit, and medical models have been shown to be vulnerable to such manipulation [8]. Third, incident response needs an account of why a system acted as it did; a model that cannot be explained cannot be audited, and a model that cannot be audited cannot be trusted after an incident. Interpretable models are sometimes a better answer than explanations of black boxes [7], and the next notebook compares both routes.

## 7. Three legal mandates

The table summarises the three instruments named in the workshop programme. Dates are those in force when the material was checked on 28 September 2026.

| Instrument | What it requires | Status for medical AI |
|---|---|---|
| **EU AI Act**, Regulation (EU) 2024/1689 [9] | Article 13: high-risk systems must be transparent enough for deployers to interpret the output and use it appropriately, with instructions for use that describe capabilities and limitations. Article 14 adds human oversight; Article 15 requires accuracy, robustness and cybersecurity, including measures against adversarial examples and data poisoning. | AI that is a safety component of, or itself, a medical device under the MDR or IVDR is high-risk through Annex I. The Digital Omnibus on AI, Regulation (EU) 2026/1744 [10], moved the start of these obligations to 2 August 2028 for Annex I systems and to 2 December 2027 for Annex III systems. |
| **FDA, software as a medical device** [14, 15, 16, 17, 18] | Clinical decision support software can fall outside the device definition only if, among other criteria, it lets the professional independently review the basis for its recommendation (FD&C Act, section 520(o)(1)(E)). The FDA, Health Canada and MHRA transparency principles ask for information on performance, limitations and the logic of the output. | The predetermined change control plan guidance is final (December 2024); the lifecycle guidance for AI-enabled device software functions was still a draft (January 2025) in September 2026; the premarket cybersecurity guidance was finalised in February 2026. The FDA list counted more than 1,600 authorised AI-enabled devices in September 2026. |
| **GDPR**, Regulation (EU) 2016/679 [11] | Article 22 restricts decisions based solely on automated processing with legal or similarly significant effects and guarantees human intervention and the right to contest. Articles 13 to 15 require meaningful information about the logic involved. | The Court of Justice has read these provisions broadly: a score on which a decision strongly relies can itself be an automated decision [12], and the logic must be explained in an intelligible way [13]. A clinician in the loop narrows Article 22, but the information duties remain. |

## Discussion

1. The model reaches roughly 97 percent accuracy. Which additional numbers should a hospital committee request before a deployment decision?
2. Who should choose the decision threshold, and on what grounds?
3. In the shortcut experiment, which internal check could have revealed the problem before the external hospital did?
4. Which of the three instruments applies to a decision support tool that a clinician can always override, and why?

## Summary

A single performance number is an insufficient basis for clinical deployment. It hides the asymmetry of errors, it says nothing about confident mistakes, it does not reveal whether probabilities are calibrated, and it cannot distinguish a model that learned the disease from one that learned the hospital. Each of these gaps is also a security gap. The next notebook, NB2, builds the toolkit that closes them.

## References

[1] W. N. Street, W. H. Wolberg, and O. L. Mangasarian, "Nuclear feature extraction for breast tumor diagnosis," in *Proc. SPIE 1905, Biomedical Image Processing and Biomedical Visualization*, 1993, pp. 861–870. doi: 10.1117/12.148698

[2] W. Wolberg, O. Mangasarian, N. Street, and W. Street, "Breast Cancer Wisconsin (Diagnostic)," UCI Machine Learning Repository, 1993. https://archive.ics.uci.edu/dataset/17

[3] C. Guo, G. Pleiss, Y. Sun, and K. Q. Weinberger, "On calibration of modern neural networks," in *Proc. 34th Int. Conf. Machine Learning (ICML)*, PMLR 70, 2017, pp. 1321–1330.

[4] J. R. Zech, M. A. Badgeley, M. Liu, A. B. Costa, J. J. Titano, and E. K. Oermann, "Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: A cross-sectional study," *PLOS Medicine*, vol. 15, no. 11, e1002683, 2018. doi: 10.1371/journal.pmed.1002683

[5] A. J. DeGrave, J. D. Janizek, and S.-I. Lee, "AI for radiographic COVID-19 detection selects shortcuts over signal," *Nature Machine Intelligence*, vol. 3, pp. 610–619, 2021. doi: 10.1038/s42256-021-00338-7

[6] R. Caruana, Y. Lou, J. Gehrke, P. Koch, M. Sturm, and N. Elhadad, "Intelligible models for healthcare: Predicting pneumonia risk and hospital 30-day readmission," in *Proc. 21st ACM SIGKDD*, 2015, pp. 1721–1730. doi: 10.1145/2783258.2788613

[7] C. Rudin, "Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead," *Nature Machine Intelligence*, vol. 1, pp. 206–215, 2019. doi: 10.1038/s42256-019-0048-x

[8] S. G. Finlayson, J. D. Bowers, J. Ito, J. L. Zittrain, A. L. Beam, and I. S. Kohane, "Adversarial attacks on medical machine learning," *Science*, vol. 363, no. 6433, pp. 1287–1289, 2019. doi: 10.1126/science.aaw4399

[9] Regulation (EU) 2024/1689 of the European Parliament and of the Council laying down harmonised rules on artificial intelligence (Artificial Intelligence Act), *Official Journal of the European Union*, L series, 12 July 2024.

[10] Regulation (EU) 2026/1744 (Digital Omnibus on AI), amending Regulation (EU) 2024/1689, published in the *Official Journal of the European Union* on 24 July 2026 and in force since 27 July 2026.

[11] Regulation (EU) 2016/679 (General Data Protection Regulation), *Official Journal of the European Union*, L 119, 4 May 2016.

[12] Court of Justice of the European Union, Case C-634/21, *SCHUFA Holding (Scoring)*, judgment of 7 December 2023.

[13] Court of Justice of the European Union, Case C-203/22, *Dun & Bradstreet Austria*, judgment of 27 February 2025.

[14] U.S. Food and Drug Administration, Health Canada, and Medicines and Healthcare products Regulatory Agency, "Transparency for Machine Learning-Enabled Medical Devices: Guiding Principles," June 2024.

[15] U.S. Food and Drug Administration, "Marketing Submission Recommendations for a Predetermined Change Control Plan for Artificial Intelligence-Enabled Device Software Functions," final guidance, December 2024.

[16] U.S. Food and Drug Administration, "Artificial Intelligence-Enabled Device Software Functions: Lifecycle Management and Marketing Submission Recommendations," draft guidance, January 2025.

[17] U.S. Food and Drug Administration, "Cybersecurity in Medical Devices: Quality Management System Considerations and Content of Premarket Submissions," final guidance, February 2026.

[18] Federal Food, Drug, and Cosmetic Act, section 520(o)(1)(E), 21 U.S.C. 360j(o)(1)(E).